In [ ]:
# import google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -lh "/content/drive/MyDrive/ECE_M214A/project_214A/"

ls: cannot access '/content/drive/MyDrive/ECE_M214A/project_214A/': No such file or directory


In [ ]:
import os
from glob import glob

base = "/content/drive/MyDrive/ECE_M214A/project_214A/"

print("Base exists?", os.path.exists(base))
if os.path.exists(base):
    print("Base contents:", os.listdir(base))

# Find wavs anywhere under that folder (recursive, case-insensitive)
wavs = glob(base + "/**/*.wav", recursive=True) + glob(base + "/**/*.WAV", recursive=True)
print("Num wav files found:", len(wavs))
print("First 10:", wavs[:10])

Base exists? False
Num wav files found: 0
First 10: []


In [ ]:
import os
import random
import numpy as np
import copy

# Set before CUDA ops for deterministic CUDA kernels.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchaudio
import librosa
from glob import glob

SEED = 0

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
# ============================================================
# Dataset paths
# ============================================================
TRAIN_DIR = "/content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/train_clean"
TEST_CLEAN_DIR = "/content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/test_clean"
TEST_NOISY_5DB_DIR = "/content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/test_snr_5db_babble"
TEST_NOISY_10DB_DIR = "/content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/test_snr_10db_babble"

# ============================================================
# Audio loading
# ============================================================

def load_audio(audio_file):
    audio, fs = torchaudio.load(audio_file)
    audio = audio.numpy().reshape(-1)
    return audio, int(fs)

In [ ]:
# ============================================================
# Feature extraction — add new extractors here
# ============================================================
import numpy as np
import librosa

def _cmvn(feat, eps=1e-8):
    mu = np.mean(feat, axis=1, keepdims=True)
    sd = np.std(feat, axis=1, keepdims=True)
    return (feat - mu) / (sd + eps)

def _safe_delta(feat, order=1):
    T = feat.shape[1]
    if T < 3:
        return np.zeros_like(feat)
    width = min(9, T)
    if width % 2 == 0:
        width -= 1
    if width < 3:
        return np.zeros_like(feat)
    return librosa.feature.delta(feat, order=order, width=width)

def extract_feature(audio, fs):
    WIN_LENGTH = 200
    HOP_LENGTH = 80
    N_FFT = 256
    N_MELS = 64
    N_MFCC = 13

    audio = np.asarray(audio, dtype=np.float32)
    if audio.size == 0:
        return np.zeros((231, 1), dtype=np.float32)

    # trim to remove low-energy leading and trailing parts of the signal
    audio, _ = librosa.effects.trim(audio, top_db=40)
    if audio.size == 0:
        return np.zeros((231, 1), dtype=np.float32)

    # ---------- PCEN stream ----------
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=fs,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        power=2.0
    )

    pcen = librosa.pcen(
        mel * (2**31),
        sr=fs,
        hop_length=HOP_LENGTH,
        time_constant=0.4
    )
    pcen = pcen - pcen.mean(axis=1, keepdims=True) # normalize (64)
    pcen_d1 = _safe_delta(pcen, order=1) # first derivative (64)
    pcen_d2 = _safe_delta(pcen, order=2) # second derivative (64)
    pcen_feat = np.vstack([pcen, pcen_d1, pcen_d2])   # 192 dims (64+64+64)

    # ---------- MFCC stream ----------
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=fs,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS
    )
    mfcc = _cmvn(mfcc) # normalize with CMVN (13)
    mfcc_d1 = _safe_delta(mfcc, order=1) # first derivative (13)
    mfcc_d2 = _safe_delta(mfcc, order=2) # second derivative (13)
    mfcc_feat = np.vstack([mfcc, mfcc_d1, mfcc_d2])   # 39 dims (13+13+13)

    feat = np.vstack([pcen_feat, mfcc_feat])          # 231 dims (192+39)
    return feat.astype(np.float32)

def extract_feature_from_file(audio_file):
    audio, fs = load_audio(audio_file)
    return extract_feature(audio, fs)

In [ ]:
# ============================================================
# Dataset & DataLoader
# ============================================================

class FeatureDataset(torch.utils.data.Dataset):
    def __init__(self, X_list, y):
        self.X = X_list          # list of (F, T) np arrays
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])  # (F, T)
        return x, int(self.y[idx]), x.shape[1]


def collate_pad(batch):
    xs, ys, lens = zip(*batch)
    B = len(xs)
    F = xs[0].shape[0]
    T_max = max(lens)
    xb = torch.zeros(B, 1, F, T_max, dtype=xs[0].dtype)
    for i, x in enumerate(xs):
        xb[i, 0, :, :x.shape[1]] = x
    return xb, torch.tensor(ys, dtype=torch.long), torch.tensor(lens, dtype=torch.long)

# ============================================================
# Data loading
# ============================================================

def get_label(file_name):
    base = os.path.splitext(os.path.basename(file_name))[0]
    return int(base.split("_")[0])


def load_dir(data_dir, desc="Loading"):
    files = sorted(glob(os.path.join(data_dir, "*.wav")))
    if not files:
        print(f"No wav files found in {data_dir}")
        return [], []
    feats, labels = [], []
    for wav in files:
        feats.append(extract_feature_from_file(wav))
        labels.append(get_label(wav))
    print(f"  {desc}: {len(feats)} files loaded")
    return feats, labels

In [ ]:
# ============================================================
# LSTM model
# ============================================================

class SimpleLSTM(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=128, num_layers=2,
            batch_first=True, bidirectional=True,
            dropout=0.2
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 10),
        )

    def forward(self, x, lengths):
        # x: (B,1,F,T) -> (B,T,F)
        x = x.squeeze(1).permute(0, 2, 1).contiguous()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)  # (B, T_max, 256)

        # ---- mean pooling over valid frames (ignore padding) ----
        B, T_max, D = out.shape
        device = out.device

        # mask: (B, T_max), True for valid timesteps
        mask = torch.arange(T_max, device=device).unsqueeze(0) < lengths.unsqueeze(1)

        # expand mask to (B, T_max, D) and zero-out padding
        mask_f = mask.unsqueeze(-1).float()
        out_sum = (out * mask_f).sum(dim=1)                       # (B, D)
        denom = mask_f.sum(dim=1).clamp(min=1.0)                  # (B, 1)
        mean = out_sum / denom                                    # (B, D)

        return self.classifier(mean)

In [ ]:
# ============================================================
# Evaluation
# ============================================================

import matplotlib.pyplot as plt
from sklearn import metrics

@torch.no_grad()
def evaluate(model, loader, device, plot_cm=False, class_names=None, title=None, save_path=None):
    """
    Returns:
      acc (float) by default.
      If plot_cm=True, also returns (acc, cm).
    """
    model.eval()
    if loader is None:
        return (0.0, None) if plot_cm else 0.0

    all_preds = []
    all_labels = []

    correct, total = 0, 0
    for xb, yb, lengths in loader:
        xb, yb, lengths = xb.to(device), yb.to(device), lengths.to(device)
        logits = model(xb, lengths)
        preds = logits.argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

        if plot_cm:
            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(yb.detach().cpu().numpy())

    acc = correct / total if total > 0 else 0.0

    if not plot_cm:
        return acc

    y_pred = np.concatenate(all_preds) if all_preds else np.array([], dtype=np.int64)
    y_true = np.concatenate(all_labels) if all_labels else np.array([], dtype=np.int64)

    cm = metrics.confusion_matrix(y_true, y_pred, labels=None)

    # Plot
    disp = metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names if class_names is not None else None
    )
    disp.plot(values_format="d")
    plt.title(title or "Confusion Matrix")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close()
    else:
        plt.show()

    return acc, cm

In [ ]:
# Training
BATCH_SIZE = 32
NUM_EPOCHS = 40
LR = 3e-4

In [ ]:
import time
import torch

# ---------------- TIMING START ----------------
t0 = time.perf_counter()

# --- Load data ---
train_feat, train_label   = load_dir(TRAIN_DIR, desc="Train")
test_feat,  test_label    = load_dir(TEST_CLEAN_DIR, desc="Test clean")
noisy5_feat, noisy5_label = load_dir(TEST_NOISY_5DB_DIR, desc="Test noisy 5dB")
noisy10_feat, noisy10_label = load_dir(TEST_NOISY_10DB_DIR, desc="Test noisy 10dB")

feat_dim = train_feat[0].shape[0]
print(f"\nFeature dim: {feat_dim}")
print(f"Train: {len(train_feat)}  |  Test clean: {len(test_feat)}  "
      f"|  Test noisy 5dB: {len(noisy5_feat)}  |  Test noisy 10dB: {len(noisy10_feat)}")

for name, flist in [("Train", train_feat), ("Test clean", test_feat),
                    ("Noisy 5dB", noisy5_feat), ("Noisy 10dB", noisy10_feat)]:
    if flist:
        lengths = [f.shape[1] for f in flist]
        print(f"  {name:10s} frames: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}")

y_train   = np.array(train_label, dtype=np.int64)
y_test    = np.array(test_label,  dtype=np.int64)
y_noisy5  = np.array(noisy5_label, dtype=np.int64)
y_noisy10 = np.array(noisy10_label, dtype=np.int64)

# --- DataLoaders ---
loader_g = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    FeatureDataset(train_feat, y_train),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pad, generator=loader_g,
)
test_loader = DataLoader(
    FeatureDataset(test_feat, y_test),
    batch_size=16, shuffle=False, collate_fn=collate_pad
)
noisy5_loader = None
if noisy5_feat:
    noisy5_loader = DataLoader(
        FeatureDataset(noisy5_feat, y_noisy5),
        batch_size=16, shuffle=False, collate_fn=collate_pad,
    )
noisy10_loader = None
if noisy10_feat:
    noisy10_loader = DataLoader(
        FeatureDataset(noisy10_feat, y_noisy10),
        batch_size=16, shuffle=False, collate_fn=collate_pad,
    )

# finish "load/features" timing (include dataloader construction)
if torch.cuda.is_available():
    torch.cuda.synchronize()
t1 = time.perf_counter()
# ---------------- TIMING: LOAD/FEATURES END ----------------


# --- Training ---
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = SimpleLSTM(input_size=feat_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)

best_clean, best_clean_ep = 0.0, -1
best_5db,  best_5db_ep  = 0.0, -1
best_10db, best_10db_ep = 0.0, -1

"""
Save the checkpoint with the best 10 dB accuracy.
You may instead choose to save the model with the best clean or 5 dB accuracy,
but keep in mind that your final score depends on strong performance on all of the test sets, especially the noisy test set.
"""
saved_checkpoint = None

for epoch in range(1, NUM_EPOCHS + 1):
    net.train()
    total_loss = 0.0
    for xb, yb, lengths in train_loader:
        xb, yb, lengths = xb.to(device), yb.to(device), lengths.to(device)
        optimizer.zero_grad()
        loss = criterion(net(xb, lengths), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    clean_acc = evaluate(net, test_loader, device)
    acc_5db  = evaluate(net, noisy5_loader, device)
    acc_10db = evaluate(net, noisy10_loader, device)

    print(f"Epoch {epoch:02d}  loss={avg_loss:.4f}  "
          f"clean={clean_acc:.4f}  5dB={acc_5db:.4f}  10dB={acc_10db:.4f}")

    if clean_acc > best_clean:
        best_clean, best_clean_ep = clean_acc, epoch
    if acc_5db > best_5db:
        best_5db, best_5db_ep = acc_5db, epoch
    if acc_10db > best_10db:
        best_10db, best_10db_ep = acc_10db, epoch
        saved_checkpoint = copy.deepcopy(net.state_dict())  # save the best checkpoint

# finish "train" timing
if torch.cuda.is_available():
    torch.cuda.synchronize()
t2 = time.perf_counter()
# ---------------- TIMING: TRAIN END ----------------


# --- Final evaluation timing (optional but nice) ---
# If you want this to reflect "best checkpoint", load it:
if saved_checkpoint is not None:
    net.load_state_dict(saved_checkpoint)

clean_final = evaluate(net, test_loader, device)
acc5_final  = evaluate(net, noisy5_loader, device)
acc10_final = evaluate(net, noisy10_loader, device)

if torch.cuda.is_available():
    torch.cuda.synchronize()
t3 = time.perf_counter()
# ---------------- TIMING: EVAL END ----------------

print(f"\nTiming:")
print(f"Load/features: {t1 - t0:.1f}s")
print(f"Train:         {t2 - t1:.1f}s")
print(f"Eval:          {t3 - t2:.1f}s")
print(f"TOTAL:         {t3 - t0:.1f}s")

print(f"\nFinal (best-ckpt) accuracies:")
print(f"clean={clean_final:.4f}  5dB={acc5_final:.4f}  10dB={acc10_final:.4f}")


No wav files found in /content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/train_clean
No wav files found in /content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/test_clean
No wav files found in /content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/test_snr_5db_babble
No wav files found in /content/drive/MyDrive/ECE_M214A/project_214A/M214_project_data/test_snr_10db_babble


IndexError: list index out of range

In [ ]:
# load the best checkpoint before testing
net = SimpleLSTM(input_size=feat_dim).to(device)
net.load_state_dict(saved_checkpoint)
# clean_acc = evaluate(net, test_loader, device)
clean_acc, cm_clean = evaluate(
    net, test_loader, device,
    plot_cm=True,
    class_names=list(range(10)),
    title="Confusion Matrix — Clean"
)
acc_5db, cm_5db = evaluate(
    net, noisy5_loader, device,
    plot_cm=True,
    class_names=list(range(10)),
    title="Confusion Matrix — Noisy 5 dB"
)
acc_10db, cm_10db = evaluate(
    net, noisy10_loader, device,
    plot_cm=True,
    class_names=list(range(10)),
    title="Confusion Matrix — Noisy 10 dB"
)
print(f"\nBest checkpoint loaded (epoch {best_10db_ep})")
print(f"Clean accuracy : {clean_acc:.4f}")
print(f"5dB accuracy   : {acc_5db:.4f}")
print(f"10dB accuracy  : {acc_10db:.4f}")